# Motorised mount client

Use the same motor client as the GUI: **notebook → client → server → serial driver**.
Select a named setup from `JazLabs.launchers.configs.motorised_mounts`:

| Setup | Controller | Axes | Position units | Server port |
|---|---|---|---|---|
| `newport_tip_tilt` | CONEX-AGAP / AG-M100D | U (yaw), V (pitch) | degrees | 50941 |
| `newport_translation` | CONEX-AGP | X | mm, if configured that way on the fitted stage | 50942 |
| `thorlabs_umc2` | UMC2 | X, Y (bays 0x21, 0x22) | native controller counts | 50943 |
| `simulation` | simulated two-axis stage | X, Y | mm | 50944 |

The Newport translation preset is for **CONEX-AGP**, not an AG-UC8 controller.
Hardware serial ports are intentionally unset. Set them in the config or supply
`--serial-port` when launching the server. Travel limits must be calibrated for the
installed mechanism; the empty config limits do not invent a travel range.

All movement, homing, and speed changes are disabled by default. Running this
notebook only connects, reads feedback, and closes its client.

In [ ]:
from copy import deepcopy
import shlex
import sys
import time

from JazLabs.hardware.MotorisedStages.MotorisedStage_Client import MotorisedStageClient
from JazLabs.launchers.config import get_named_config, load_config

## Select the mount

Use a remote `HOST_OVERRIDE` for direct access to the hardware computer, or set
`USE_BRIDGE = True` for an already running local bridge.
`SERIAL_PORT_OVERRIDE` only changes the printed server launch command; serial
communication is always owned by the server, not this notebook.

In [ ]:
CONFIG = "motorised_mounts"
MOUNT_NAME = "newport_tip_tilt"  # newport_translation, thorlabs_umc2, or simulation
USE_BRIDGE = False
HOST_OVERRIDE = None
COMMAND_PORT_OVERRIDE = None
SERVER_COMMAND_PORT_OVERRIDE = None
SERIAL_PORT_OVERRIDE = None  # e.g. "COM5" or "/dev/ttyUSB0", on the hardware computer

settings = deepcopy(get_named_config(load_config(CONFIG), "MOTOR_SERVERS", MOUNT_NAME))
SERVER_COMMAND_PORT = SERVER_COMMAND_PORT_OVERRIDE if SERVER_COMMAND_PORT_OVERRIDE is not None else settings["command_port"]
if USE_BRIDGE:
    default_host = settings.get("local_host", "127.0.0.1")
    default_port = settings["local_command_port"]
else:
    default_host = settings.get("client_host", "127.0.0.1")
    default_port = SERVER_COMMAND_PORT
HOST = HOST_OVERRIDE or default_host
COMMAND_PORT = COMMAND_PORT_OVERRIDE if COMMAND_PORT_OVERRIDE is not None else default_port
AXIS = settings["default_axis"]
RELATIVE_STEP = settings["relative_step"]
ABSOLUTE_TARGET = None  # Set an explicit target in the displayed units before moving.
ENABLE_MOTION = False
HOME_AXIS = False
NEW_SPEED = None
MOVE_TIMEOUT = settings["stage_kwargs"].get("move_timeout", 60.0)
print(MOUNT_NAME, "→", HOST, COMMAND_PORT, "·", AXIS, "·", settings["expected_units"])

## Start the matching server

Run the printed command in a **separate terminal on the hardware computer**.
Use that computer's Python environment and serial-port name. This cell only
prints a command; it does not start a server or open hardware. The command uses
POSIX shell quoting; adjust quoting for your shell if needed.

Start the GUI separately with `jazlabs-view-motor --config motorised_mounts
--name NAME`. Add `--host ADDRESS` for a remote direct server, or `--bridge` to
use the local bridge. No launcher starts another instrument process implicitly.
Set `USE_BRIDGE = True` to print a bridge command as well; replace its hardware
computer address or save `remote_host` in the shared preset.

In [ ]:
server_command = [
    sys.executable, "-m", "JazLabs.launchers.launch_motorised_stage_server",
    "--config", CONFIG, "--name", MOUNT_NAME,
    "--command-port", str(SERVER_COMMAND_PORT),
]
if SERIAL_PORT_OVERRIDE is not None:
    server_command += ["--serial-port", SERIAL_PORT_OVERRIDE]
print(shlex.join(server_command))
if settings["stage_type"] != "simulation" and not (SERIAL_PORT_OVERRIDE or settings["stage_kwargs"].get("port")):
    print("Before launching: set stage_kwargs.port in the config or add --serial-port YOUR_SERIAL_PORT.")
print("For a remote client, also bind the server to its trusted lab interface with --host ADDRESS.")
if USE_BRIDGE:
    bridge_command = [
        sys.executable, "-m", "JazLabs.launchers.launch_motorised_stage_bridge",
        "--config", CONFIG, "--name", MOUNT_NAME,
        "--remote-host", settings.get("remote_host") or "HARDWARE_COMPUTER_ADDRESS",
        "--remote-command-port", str(SERVER_COMMAND_PORT),
        "--local-command-port", str(COMMAND_PORT),
    ]
    print("On the control computer, after the server is running:")
    print(shlex.join(bridge_command))


## Connect and check the server

The client waits for driver startup, verifies the controller type and units, and
reads every reported axis. It never assumes a controller is homed just because
a home request was sent. Close this client before selecting another mount.

In [ ]:
if "mount" in globals():
    mount.close()
mount = MotorisedStageClient(host=HOST, command_port=COMMAND_PORT, timeout_ms=1000)
try:
    deadline = time.monotonic() + 5.0
    while True:
        snapshot = mount.request("state")
        if snapshot.get("error"):
            raise RuntimeError(snapshot["error"])
        if snapshot["connected"]:
            break
        if time.monotonic() >= deadline:
            raise TimeoutError("The server has not connected to its stage driver.")
        time.sleep(0.1)
    properties = mount.GetProperties()
    if properties["stage_type"] != settings["stage_type"]:
        raise RuntimeError(f"Expected {settings['stage_type']}, connected to {properties['stage_type']}")
    capabilities = {axis["id"]: axis for axis in mount.GetMotionCapabilities()}
    if AXIS not in capabilities:
        raise ValueError(f"Axis {AXIS} is unavailable; reported axes: {list(capabilities)}")
    for axis, spec in capabilities.items():
        if spec["units"] != settings["expected_units"]:
            raise RuntimeError(f"Unexpected units on {axis}: {spec['units']}")
    print(properties)
    display(list(capabilities.values()))
except Exception:
    mount.close()
    raise

In [ ]:
axis_feedback = []
for axis, spec in capabilities.items():
    state = mount.GetAxisState(axis)
    axis_feedback.append({
        "axis": axis, "position": state["position"], "units": spec["units"],
        "homed": state["homed"] if spec["home"] else "not required (absolute sensor)",
        "homing": state["homing"], "moving": state["moving"],
        "enabled": state["enabled"], "fault": state["fault"], "feedback_age_s": state["age"],
    })
display(axis_feedback)

## Wait for confirmed completion

A movement acknowledgement means the request was submitted, not that the stage
has arrived. This helper polls feedback and surfaces faults, stop state, stale
feedback, and timeouts. A communication timeout can leave a move's outcome
unknown; inspect the state or request a stop rather than blindly retrying it.

In [ ]:
def wait_for_axis(axis, target=None, homing=False, timeout=MOVE_TIMEOUT):
    started = time.monotonic()
    spec = capabilities[axis]
    tolerance = spec.get("tolerance", 2 * spec["resolution"])
    while time.monotonic() - started < timeout:
        snapshot = mount.request("state")
        if snapshot.get("error"):
            raise RuntimeError(snapshot["error"])
        if snapshot["stopped"] or snapshot["stop_pending"]:
            raise RuntimeError("Motion is stopped; inspect the stage before explicitly resuming.")
        state = snapshot["axes"][axis]
        if not snapshot["connected"] or state["age"] >= 1.0:
            raise RuntimeError("Feedback is unavailable or stale.")
        busy = state["moving"] or state["homing"] or state.get("commanded", False)
        reached = state["homed"] if homing else abs(state["position"] - target) <= tolerance
        if not busy and reached and time.monotonic() - started >= 0.2:
            return state
        time.sleep(0.05)
    raise TimeoutError(f"{axis} did not complete within {timeout:g} s")

## Optional homing and speed

Set `ENABLE_MOTION = True` and `HOME_AXIS = True` only when ready to home the
selected axis. AGAP has absolute sensors and requires no homing. If feedback
reports an axis disabled, explicitly enable it with `mount.EnableAxis(AXIS)`
before homing or moving. The current serial drivers do not expose physical-unit
speed settings; `NEW_SPEED` is only applied when the driver advertises support.

In [ ]:
if ENABLE_MOTION and HOME_AXIS:
    if not capabilities[AXIS]["home"]:
        print("Absolute sensor: homing is not required.")
    elif mount.GetAxisState(AXIS)["homed"]:
        print("Axis is already homed; no home request sent.")
    else:
        mount.HomeAxis(AXIS)
        print(wait_for_axis(AXIS, homing=True))
else:
    print("No homing requested.")

if ENABLE_MOTION and NEW_SPEED is not None:
    if not capabilities[AXIS]["speed"]:
        raise NotImplementedError("This driver does not expose speed configuration.")
    mount.SetSpeed(AXIS, NEW_SPEED)
else:
    print("No speed change requested.")

## Relative move

The default steps are 0.005° for tip/tilt, 0.01 mm for the AGP translation preset,
and 100 native counts for UMC2. Check that those distances suit your mechanism.
The UMC2 count-to-distance conversion depends on the fitted stages and is not
assumed by this config.

In [ ]:
if ENABLE_MOTION:
    start_position = mount.GetAxisState(AXIS)["position"]
    resolution = capabilities[AXIS]["resolution"]
    target = round((start_position + RELATIVE_STEP) / resolution) * resolution
    mount.request("relative", axis=AXIS, value=RELATIVE_STEP)
    print(wait_for_axis(AXIS, target=target))
else:
    print("Relative move disabled. Set ENABLE_MOTION = True to send it.")

## Absolute move

Enter `ABSOLUTE_TARGET` in degrees, configured AGP stage units, or native UMC2
counts as appropriate. These are stage coordinates, not an assumed mount centre.

In [ ]:
if ENABLE_MOTION and ABSOLUTE_TARGET is not None:
    resolution = capabilities[AXIS]["resolution"]
    target = round(float(ABSOLUTE_TARGET) / resolution) * resolution
    mount.MoveTo(AXIS, ABSOLUTE_TARGET)
    print(wait_for_axis(AXIS, target=target))
else:
    print("Absolute move disabled. Set ENABLE_MOTION and an explicit ABSOLUTE_TARGET.")

## Stop or resume explicitly

`StopAll` interrupts all axes and latches motion off. Resume only after inspecting
the stage. These are software commands and cannot replace a physical emergency
stop when the communication link is unavailable. The lines are commented so
**Run All** does not stop or resume a mount unexpectedly.

In [ ]:
# mount.StopAll()
# display(mount.request("state"))
# mount.request("resume")

## Disconnect

Closing this synchronous notebook client does not stop the motor or shut down
its server. Request `StopAll()` first if you intend to stop active motion. A GUI
client requests a stop when its window closes.

In [ ]:
mount.close()
print("Notebook client disconnected; the motor server remains running.")